In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy qiskit-ibm-catalog scikit-learn
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Singularity Machine Learning - Classification: A Qiskit Function by Multiverse Computing

> **Note:** * Funcțiile Qiskit sunt o funcționalitate experimentală disponibilă doar utilizatorilor IBM Quantum&reg; Premium Plan, Flex Plan și On-Prem (prin IBM Quantum Platform API) Plan. Acestea se află în stare de previzualizare și pot suferi modificări.

## Prezentare generală
Cu funcția „Singularity Machine Learning - Classification" poți rezolva probleme reale de machine learning pe hardware cuantic fără a necesita expertiză cuantică. Această funcție applicație, bazată pe metode ensemble, este un clasificator hibrid. Utilizează metode clasice precum boosting, bagging și stacking pentru antrenarea inițială a ensemble-ului. Ulterior, se folosesc algoritmi cuantici precum variational quantum eigensolver (VQE) și quantum approximate optimization algorithm (QAOA) pentru a îmbunătăți diversitatea, capacitățile de generalizare și complexitatea globală a ensemble-ului antrenat.

Spre deosebire de alte soluții de quantum machine learning, această funcție este capabilă să gestioneze seturi de date la scară largă cu milioane de exemple și caracteristici, fără a fi limitată de numărul de qubiți din QPU-ul țintă. Numărul de qubiți determină doar dimensiunea ensemble-ului care poate fi antrenat. De asemenea, este extrem de flexibilă și poate fi utilizată pentru a rezolva probleme de clasificare într-o gamă largă de domenii, inclusiv finanțe, sănătate și securitate cibernetică.
Obține în mod consistent acuratețe ridicată pe probleme clasic dificile, care implică seturi de date de înaltă dimensionalitate, zgomotoase și dezechilibrate.
![Cum funcționează](../docs/images/guides/multiverse-computing-singularity/how_it_works.avif)
Este construită pentru:
1. Ingineri și oameni de știință din domeniul datelor din companii care caută să-și îmbunătățească oferta tehnologică integrând quantum machine learning în produsele și serviciile lor,
2. Cercetători din laboratoare de cercetare cuantică care explorează aplicații de quantum machine learning și doresc să valorifice calculul cuantic pentru sarcini de clasificare, și
3. Studenți și profesori din instituții de învățământ în cursuri precum machine learning, care caută să demonstreze avantajele calculului cuantic.

Următorul exemplu prezintă diversele funcționalități ale acesteia, inclusiv `create`, `list`, `fit` și `predict`, și demonstrează utilizarea sa într-o problemă sintetică ce cuprinde două semicercuri interconectate, o problemă notorie dificilă din cauza graniței sale de decizie neliniare.


## Descrierea funcției
Această funcție Qiskit permite utilizatorilor să rezolve probleme de clasificare binară folosind clasificatorul ensemble îmbunătățit cuantic al Singularity. În spatele scenei, folosește o abordare hibridă pentru a antrena clasic un ensemble de clasificatoare pe setul de date etichetat, iar apoi îl optimizează pentru diversitate maximă și generalizare folosind Quantum Approximate Optimization Algorithm (QAOA) pe QPU-urile IBM&reg;. Printr-o interfață prietenoasă cu utilizatorul, poți configura un clasificator conform cerințelor tale, îl antrenezi pe setul de date dorit și îl utilizezi pentru a face predicții pe un set de date nevăzut anterior.

Pentru a rezolva o problemă generică de clasificare:
1. Preprocesează setul de date și împarte-l în seturi de antrenare și testare. Opțional, poți împărți în continuare setul de antrenare în seturi de antrenare și validare. Acest lucru poate fi realizat folosind [scikit-learn](https://scikit-learn.org/1.5/modules/generated/sklearn.model_selection.train_test_split.html).
2. Dacă setul de antrenare este dezechilibrat, îl poți reeșantiona pentru a echilibra clasele folosind [imbalanced-learn](https://imbalanced-learn.org/stable/introduction.html#problem-statement-regarding-imbalanced-data-sets).
3. Încarcă seturile de antrenare, validare și testare separat în stocarea funcției folosind metoda `file_upload` a catalogului, pasând calea relevantă de fiecare dată.
4. Inițializează clasificatorul cuantic folosind acțiunea `create` a funcției, care acceptă hiperparametri precum numărul și tipurile de algoritmi de învățare, regularizarea (valoarea lambda) și opțiunile de optimizare, inclusiv numărul de straturi, tipul de optimizer clasic, Backend-ul cuantic și altele.
5. Antrenează clasificatorul cuantic pe setul de antrenare folosind acțiunea `fit` a funcției, pasând setul de antrenare etichetat și setul de validare dacă este aplicabil.
6. Realizează predicții pe setul de testare nevăzut anterior folosind acțiunea `predict` a funcției.
## Abordare bazată pe acțiuni
Funcția folosește o abordare bazată pe acțiuni. O poți gândi ca un mediu virtual în care folosești acțiuni pentru a efectua sarcini sau a-i schimba starea. În prezent, oferă următoarele acțiuni: [list](#1-list), [create](#2-create), [delete](#3-delete), [fit](#4-fit), [predict](#5-predict), [fit_predict](#6-fit-predict) și [create_fit_predict](#7-create-fit-predict). Următorul exemplu demonstrează acțiunea `create_fit_predict`.

In [1]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# load function
singularity = catalog.load("multiverse/singularity")

## Examples

### Classify a dataset

In this example, you'll use the "Singularity Machine Learning - Classification" function to classify a dataset consisting of two interleaving, moon-shaped half-circles. The dataset is synthetic, two-dimensional, and labeled with binary labels. It is created to be challenging for algorithms such as centroid-based clustering and linear classification.

![Moons dataset](../docs/images/guides/multiverse-computing-singularity/moon_shaped.avif)

Through this process, you'll learn how to create the classifier, fit it to the training data, use it to predict on the test data, and delete the classifier when you're finished.

Before starting, you need to install [scikit-learn](https://scikit-learn.org/). Install it using the following command:

```sh
python3 -m pip install scikit-learn
```

Perform the following steps:

1) Create the synthetic dataset using the [`make_moons`](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_moons.html) function from [scikit-learn](https://scikit-learn.org/).
2) Upload the generated synthetic dataset to the [shared data directory](https://qiskit.github.io/qiskit-serverless/function_features/experimental/manage_data_directory.html).
3) Create the quantum-enhanced classifier using the [`create`](/docs/api/functions/multiverse-computing-singularity#create) action.
4) Enlist your classifiers using the [`list`](/docs/api/functions/multiverse-computing-singularity#list) action.
5) Train the classifier on the train data using the [`fit`](/docs/api/functions/multiverse-computing-singularity#fit) action.
6) Use the trained classifier to predict on the test data using the [`predict`](/docs/api/functions/multiverse-computing-singularity#predict) action.
7) Delete the classifier using the [`delete`](/docs/api/functions/multiverse-computing-singularity#delete) action.
8) Clean up after you're done.

**Step 1.** Import the necessary modules and generate the synthetic dataset, then split it into training and test datasets.

In [2]:
# import the necessary modules for this example
import os
import tarfile
import numpy as np

# Import the make_moons and the train_test_split functions from scikit-learn
# to create a synthetic dataset and split it into training and test datasets
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

# generate the synthetic dataset
X, y = make_moons(n_samples=10000)

# split the data into training and test datasets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# print the first 10 samples of the training dataset
print("Features:", X_train[:10, :])
print("Targets:", y_train[:10])

Features: [[ 0.84757037 -0.48831433]
 [ 0.98132552  0.19235443]
 [-0.71626723  0.6978261 ]
 [ 1.18957848 -0.48186557]
 [ 0.52118982 -0.37791846]
 [ 0.81115408  0.58483251]
 [ 0.48706462  0.87336593]
 [-0.81880144  0.57407682]
 [ 1.67335408 -0.23932015]
 [ 0.50181306  0.8649761 ]]
Targets: [1 0 0 1 1 0 0 0 1 0]


### 1. List

Acțiunea `list` recuperează toți clasificatorii stocați în format `*.pkl.tar` din directorul de date partajat. Poți accesa și conținutul acestui director folosind metoda `catalog.files()`. În general, acțiunea list caută fișiere cu extensia `*.pkl.tar` în directorul de date partajat și le returnează în format de listă.
#### Intrări
|     Nume    |    Tip     | Descriere |   Obligatoriu  |
|-------------|-------------|-------------|-------------|
| `action` | `str` | Numele acțiunii dintre `create`, `list`, `fit`, `predict`, `fit_predict`, `create_fit_predict` și `delete`. | Da |

#### Utilizare

In [3]:
def make_tarfile(file_path, tar_file_name):
    with tarfile.open(tar_file_name, "w") as tar:
        tar.add(file_path, arcname=os.path.basename(file_path))


# save the training and test datasets on your local disk
np.save("X_train.npy", X_train)
np.save("y_train.npy", y_train)
np.save("X_test.npy", X_test)
np.save("y_test.npy", y_test)

# create tar files for the datasets
make_tarfile("X_train.npy", "X_train.npy.tar")
make_tarfile("y_train.npy", "y_train.npy.tar")
make_tarfile("X_test.npy", "X_test.npy.tar")
make_tarfile("y_test.npy", "y_test.npy.tar")

# upload the datasets to the shared data directory
catalog.file_upload("X_train.npy.tar", singularity)
catalog.file_upload("y_train.npy.tar", singularity)
catalog.file_upload("X_test.npy.tar", singularity)
catalog.file_upload("y_test.npy.tar", singularity)

# view/enlist the uploaded files in the shared data directory
print(catalog.files(singularity))

['X_test.npy.tar', 'X_train.npy.tar', 'y_test.npy.tar', 'y_train.npy.tar']


### 2. Create

Acțiunea `create` creează un clasificator de tipul `quantum_classifier` specificat, folosind parametrii furnizați, și îl salvează în directorul de date partajate.

> **Note:** Funcția acceptă momentan doar `QuantumEnhancedEnsembleClassifier`.
#### Intrări
|     Nume    |    Tip     | Descriere | Obligatoriu  | Implicit |
|-------------|------------|-----------|--------------|----------|
| `action` | `str` | Numele acțiunii, dintre `create`, `list`, `fit`, `predict`, `fit_predict`, `create_fit_predict` și `delete`. | Da | - |
| `name` | `str` | Numele clasificatorului quantum, de ex. `spam_classifier`. | Da | - |
| `instance` | `str` | Instanța IBM. | Da | - |
| `backend_name` | `str` | Resursa de calcul IBM. Implicit este `None`, ceea ce înseamnă că va fi folosit Backend-ul cu cele mai puține job-uri în așteptare. | Nu | `None` |
| `quantum_classifier` | `str` | Tipul clasificatorului quantum, adică `QuantumEnhancedEnsembleClassifier`. | Nu | `QuantumEnhancedEnsembleClassifier` |
| `num_learners` | `integer` | Numărul de learner-e din ansamblu. | Nu | `10` |
| `learners_types` | `list` | Tipurile de learner-e. Printre tipurile acceptate se numără: `DecisionTreeClassifier`, `GaussianNB`, `KNeighborsClassifier`, `MLPClassifier` și `LogisticRegression`. Detalii suplimentare despre fiecare pot fi găsite în [documentația scikit-learn](https://scikit-learn.org/stable/supervised_learning.html). | Nu | `[DecisionTreeClassifier]` |
| `learners_proportions` | `list` | Proporțiile fiecărui tip de learner în ansamblu. | Nu | `[1.0]` |
| `learners_options` | `list` | Opțiuni pentru fiecare tip de learner din ansamblu. Pentru o listă completă a opțiunilor corespunzătoare tipului/tipurilor de learner ales/alese, consultă [documentația scikit-learn](https://scikit-learn.org/stable/supervised_learning.html). | Nu | `[{"max_depth": 3, "splitter": "random", "class_weight": None}]` |
| `regularization_type` | `str` sau `list` | Tipul/tipurile de regularizare care vor fi utilizate: `onsite` sau `alpha`. `onsite` controlează termenul onsite, unde valori mai mari duc la ansambluri mai rare. `alpha` controlează echilibrul dintre termenii de interacțiune și cei onsite, unde valori mai mici duc la ansambluri mai rare. Dacă se furnizează o listă, modelele vor fi antrenate pentru fiecare tip, iar cel mai performant va fi selectat. | Nu | `onsite` |
| `regularization` | `str` sau `float` sau `list` | Valoarea de regularizare. Cuprinsă între `0` și `+inf` dacă regularization_type este `onsite`. Cuprinsă între `0` și `1` dacă regularization_type este `alpha`. Dacă este setat la `auto`, se folosește auto-regularizarea — parametrul optim de regularizare este găsit prin căutare binară, cu raportul dorit de clasificatori selectați față de totalul clasificatorilor (`regularization_desired_ratio`) și limita superioară a parametrului de regularizare (`regularization_upper_bound`). Dacă se furnizează o listă, modelele vor fi antrenate pentru fiecare valoare, iar cel mai performant va fi selectat. | Nu | `0.01` |
| `regularization_desired_ratio` | `float` sau `list` | Raportul/rapoartele dorite de clasificatori selectați față de totalul clasificatorilor pentru auto-regularizare. Dacă se furnizează o listă, modelele vor fi antrenate pentru fiecare raport, iar cel mai performant va fi selectat. | Nu | `0.75` |
| `regularization_upper_bound` | `float` sau `list` | Limita/limitele superioare ale parametrului de regularizare când se folosește auto-regularizarea. Dacă se furnizează o listă, modelele vor fi antrenate pentru fiecare limită superioară, iar cel mai performant va fi selectat. | Nu | `200` |
| `weight_update_method` | `str` | Metoda de actualizare a ponderilor eșantioanelor, dintre `logarithmic` și `quadratic`. | Nu | `logarithmic` |
| `sample_scaling` | `boolean` | Dacă scalarea eșantioanelor trebuie aplicată. | Nu | `False` |
| `prediction_scaling` | `float` | Factorul de scalare pentru predicții. | Nu | `None` |
| `optimizer_options` | `dictionary` | Opțiunile optimizatorului QAOA. O listă cu opțiunile disponibile este prezentată mai jos în această documentație. | Nu | ... |
| `voting` | `str` | Folosește votul majoritar (`hard`) sau media probabilităților (`soft`) pentru agregarea predicțiilor/probabilităților learner-elor. | Nu | `hard` |
| `prob_threshold` | `float` | Pragul optim de probabilitate. | Nu | `0.5` |
| `random_state` | `integer` | Controlează aleatorismul pentru reproductibilitate. | Nu | `None` |


- În plus, `optimizer_options` sunt enumerate după cum urmează:

|     Nume    |    Tip     | Descriere | Obligatoriu  | Implicit |
|-------------|------------|-----------|--------------|----------|
| `num_solutions` | `integer` | Numărul de soluții | Nu | `1024` |
| `reps` | `integer` | Numărul de repetiții | Nu | `4` |
| `sparsify` | `float` | Pragul de sparsificare | Nu | `0.001` |
| `theta` | `float` | Valoarea inițială a lui theta, un parametru variațional al QAOA | Nu | `None` |
| `simulator` | `boolean` | Dacă să fie folosit un simulator sau un QPU | Nu | `False` |
| `classical_optimizer` | `str` | Numele optimizatorului clasic pentru QAOA. Toți solverii oferiți de SciPy, enumerați [aici](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.minimize.html#scipy.optimize.minimize), pot fi utilizați. Va trebui să setezi `classical_optimizer_options` corespunzător | Nu | `COBYLA` |
| `classical_optimizer_options` | `dictionary` | Opțiunile optimizatorului clasic. Pentru o listă completă a opțiunilor disponibile, consultă [documentația SciPy](https://docs.scipy.org/doc/scipy/reference/) | Nu | `{"maxiter": 60}` |
| `optimization_level` | `integer` | Adâncimea Circuit-ului QAOA | Nu | `3` |
| `num_transpiler_runs` | `integer` | Numărul de rulări ale Transpiler-ului | Nu | `30` |
| `pass_manager_options` | `dictionary` | Opțiuni pentru generarea pass manager-ului preset | Nu | `{"approximation_degree": 1.0}` |
| `estimator_options` | `dictionary` | Opțiunile Estimator-ului. Pentru o listă completă a opțiunilor disponibile, consultă [documentația Qiskit Runtime Client](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options) | Nu | `None` |
| `sampler_options` | `dictionary` | Opțiunile Sampler-ului. Pentru o listă completă a opțiunilor disponibile, consultă [documentația Qiskit Runtime Client](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-sampler-options) | Nu | `None` |

- `estimator_options` implicite sunt:

|     Nume    |    Tip     | Valoare  |
|-------------|------------|----------|
| `default_shots` | `integer` | `1024` |
| `resilience_level` | `integer` | `2` |
| `twirling` | `dictionary` | `{"enable_gates": True}` |
| `dynamical_decoupling` | `dictionary` | `{"enable": True}` |
| `resilience_options` | `dictionary` | `{"zne_mitigation": False, "zne": {"amplifier": "pea", "noise_factors": [1.0, 1.3, 1.6], "extrapolator": ["linear", "polynomial_degree_2", "exponential"],}}` |

- `sampler_options` implicite sunt:

|     Nume    |    Tip     | Valoare |
|-------------|------------|---------|
| `default_shots` | `integer` | `1024` |
| `resilience_level` | `integer` | `1` |
| `twirling` | `dictionary` | `{"enable_gates": True}` |
| `dynamical_decoupling` | `dictionary` | `{"enable": True}` |

#### Utilizare

In [4]:
job = singularity.run(
    action="create",
    name="my_classifier",
    num_learners=10,
    learners_types=[
        "DecisionTreeClassifier",
        "KNeighborsClassifier",
    ],
    learners_proportions=[0.5, 0.5],
    learners_options=[{}, {}],
    regularization=0.01,
    weight_update_method="logarithmic",
    sample_scaling=True,
    optimizer_options={"simulator": True},
    voting="soft",
    prob_threshold=0.5,
)

print(job.result())

{'status': 'ok', 'message': 'Classifier created.', 'data': {}, 'metadata': {'resource_usage': {}}}


In [5]:
# list available classifiers using the list action
job = singularity.run(action="list")

print(job.result())

# you can also find your classifiers in the shared data directory with a *.pkl.tar extension
print(catalog.files(singularity))

{'status': 'ok', 'message': 'Classifiers listed.', 'data': {'classifiers': ['my_classifier']}, 'metadata': {'resource_usage': {}}}


['X_test.npy.tar', 'X_train.npy.tar', 'my_classifier.pkl.tar', 'y_test.npy.tar', 'y_train.npy.tar']


#### Validări
- `name`:
    - Numele trebuie să fie unic, un șir de maximum 64 de caractere.
    - Poate conține doar caractere alfanumerice și liniuțe de subliniere.
    - Trebuie să înceapă cu o literă și nu poate să se termine cu o liniuță de subliniere.
    - Un clasificator cu același nume trebuie să existe deja în directorul de date partajate.
### 4. Fit

Acțiunea `fit` antrenează un clasificator folosind datele de antrenament furnizate.

#### Intrări
|     Nume    |    Tip     | Descriere |   Obligatoriu  |
|-------------|------------|-----------|----------------|
| `action`    | `str`    | Numele acțiunii. Trebuie să fie `fit`. | Da |
| `name`      | `str`    | Numele clasificatorului de antrenat. | Da |
| `X`         | `array` sau `list` sau `str` | Datele de antrenament. Poate fi un array NumPy, o listă sau un șir de caractere care referențiază un fișier din directorul de date partajate. | Da |
| `y`         | `array` sau `list` sau `str` | Valorile țintă pentru antrenament. Poate fi un array NumPy, o listă sau un șir de caractere care referențiază un fișier din directorul de date partajate. | Da |
| `fit_params`| `dictionary`| Parametri suplimentari de transmis metodei `fit` a clasificatorului. | Nu |

##### fit_params
|     Nume    |    Tip     | Descriere |   Obligatoriu  |   Implicit   |
|-------------|------------|-----------|----------------|--------------|
| `validation_data` | `tuple` | Datele și etichetele de validare. | Nu | `None` |
| `pos_label` | `integer` sau `str` | Eticheta de clasă care va fi mapată la 1. | Nu | `None` |
| `optimization_data` | `str` | Setul de date pe care să fie optimizat ansamblul. Poate fi unul dintre: `train`, `validation`, `both`. | Nu | `train` |

#### Utilizare

In [6]:
job = singularity.run(
    action="fit",
    name="my_classifier",
    X="X_train.npy",  # you do not need to specify the tar extension
    y="y_train.npy",  # you do not need to specify the tar extension
)

print(job.result())

{'status': 'ok', 'message': 'Classifier fitted.', 'data': {}, 'metadata': {'resource_usage': {'RUNNING: MAPPING': {'CPU_TIME': 13.655871629714966}, 'RUNNING: WAITING_QPU': {'CPU_TIME': 54.688621282577515}, 'RUNNING: POST_PROCESSING': {'CPU_TIME': 56.92286920547485}, 'RUNNING: EXECUTING_QPU': {'QPU_TIME': 57.92738223075867}}}}


#### Validări
- `name`:
    - Numele trebuie să fie unic, un șir de caractere de până la 64 de caractere.
    - Poate conține doar caractere alfanumerice și underscore-uri.
    - Trebuie să înceapă cu o literă și nu poate să se termine cu un underscore.
    - Un clasificator cu același nume trebuie să existe deja în directorul de date partajate.
### 5. Predict

Acțiunea `predict` este folosită pentru a obține predicții tari și moi (probabilități).

#### Intrări
|     Nume    |    Tip     | Descriere |   Obligatoriu  |
|-------------|-------------|-------------|-------------|
| `action`    | `str`    | Numele acțiunii. Trebuie să fie `predict`. | Da |
| `name`      | `str`    | Numele clasificatorului care va fi folosit. | Da |
| `X`         | `array` sau `list` sau `str` | Datele de test. Poate fi un array NumPy, o listă sau un șir de caractere care referențiază un nume de fișier din directorul de date partajate. | Da |
| `options["out"]` | `str` | Numele fișierului JSON de ieșire în care să fie salvate predicțiile în directorul de date partajate. Dacă nu este furnizat, predicțiile sunt returnate în rezultatul job-ului. | Nu |

#### Utilizare

In [7]:
job = singularity.run(
    action="predict",
    name="my_classifier",
    X="X_test.npy",  # you do not need to specify the tar extension
)

result = job.result()

print("Action result status: ", result["status"])
print("Action result message: ", result["message"])
print("Predictions (first five results):", result["data"]["predictions"][:5])
print(
    "Probabilities (first five results):", result["data"]["probabilities"][:5]
)

Action result status:  ok
Action result message:  Classifier predicted.
Predictions (first five results): [0, 0, 1, 0, 1]
Probabilities (first five results): [[1.0, 0.0], [1.0, 0.0], [0.0, 1.0], [1.0, 0.0], [0.0, 1.0]]


#### Validări
- `name`:
    - Numele trebuie să fie unic, un șir de caractere de până la 64 de caractere.
    - Poate conține doar caractere alfanumerice și underscore-uri.
    - Trebuie să înceapă cu o literă și nu poate să se termine cu un underscore.
    - Un clasificator cu același nume trebuie să existe deja în directorul de date partajate.
- `options["out"]`:
    - Numele fișierului trebuie să fie unic, un șir de caractere de până la 64 de caractere.
    - Poate conține doar caractere alfanumerice și underscore-uri.
    - Trebuie să înceapă cu o literă și nu poate să se termine cu un underscore.
    - Trebuie să aibă extensia `.json`.
### 6. Fit-predict

Acțiunea `fit_predict` antrenează un clasificator folosind datele de antrenament și apoi îl utilizează pentru a obține predicții tari și moi (probabilități).

#### Intrări
|     Nume    |    Tip     | Descriere |   Obligatoriu  |
|-------------|-------------|-------------|-------------|
| `action`    | `str`    | Numele acțiunii. Trebuie să fie `fit_predict`. | Da |
| `name`      | `str`    | Numele clasificatorului care va fi folosit. | Da |
| `X_train`   | `array` sau `list` sau `str` | Datele de antrenament. Poate fi un array NumPy, o listă sau un șir de caractere care referențiază un nume de fișier din directorul de date partajate. | Da |
| `y_train`   | `array` sau `list` sau `str` | Valorile țintă de antrenament. Poate fi un array NumPy, o listă sau un șir de caractere care referențiază un nume de fișier din directorul de date partajate. | Da |
| `X_test`    | `array` sau `list` sau `str` | Datele de test. Poate fi un array NumPy, o listă sau un șir de caractere care referențiază un nume de fișier din directorul de date partajate. | Da |
| `fit_params`| `dictionary`| Parametri suplimentari care vor fi pasați metodei `fit` a clasificatorului. | Nu |
| `options["out"]` | `str` | Numele fișierului JSON de ieșire în care să fie salvate predicțiile în directorul de date partajate. Dacă nu este furnizat, predicțiile sunt returnate în rezultatul job-ului. | Nu |

#### Utilizare

In [8]:
job = singularity.run(
    action="delete",
    name="my_classifier",
)

# or you can delete from the shared data directory
# catalog.file_delete("my_classifier.pkl.tar", singularity)

print(job.result())

{'status': 'ok', 'message': 'Classifier deleted.', 'data': {}, 'metadata': {'resource_usage': {}}}


#### Validări
- `name`:
    - Numele trebuie să fie unic, un șir de caractere de până la 64 de caractere.
    - Poate conține doar caractere alfanumerice și underscore-uri.
    - Trebuie să înceapă cu o literă și nu poate să se termine cu un underscore.
    - Un clasificator cu același nume trebuie să existe deja în directorul de date partajate.

- `options["out"]`:
    - Numele fișierului trebuie să fie unic, un șir de caractere de până la 64 de caractere.
    - Poate conține doar caractere alfanumerice și underscore-uri.
    - Trebuie să înceapă cu o literă și nu poate să se termine cu un underscore.
    - Trebuie să aibă extensia `.json`.
### 7. Create-fit-predict

Acțiunea `create_fit_predict` creează un clasificator, îl antrenează folosind datele de antrenament furnizate și apoi îl utilizează pentru a obține predicții tari și moi (probabilități).

#### Intrări
| Nume | Tip | Descriere | Obligatoriu |
|------|------|-------------|-----------|
| `action` | `str` | Numele acțiunii, dintre `create`, `list`, `fit`, `predict`, `fit_predict`, `create_fit_predict` și `delete`. | Da |
| `name` | `str` | Numele clasificatorului care va fi folosit. | Da |
| `quantum_classifier` | `str` | Tipul clasificatorului, adică `QuantumEnhancedEnsembleClassifier`. Implicit este `QuantumEnhancedEnsembleClassifier`. | Nu |
| `X_train` | `array` sau `list` sau `str` | Datele de antrenament. Poate fi un array NumPy, o listă sau un șir de caractere care referențiază un nume de fișier din directorul de date partajate. | Da |
| `y_train` | `array` sau `list` sau `str` | Valorile țintă de antrenament. Poate fi un array NumPy, o listă sau un șir de caractere care referențiază un nume de fișier din directorul de date partajate. | Da |
| `X_test` | `array` sau `list` sau `str` | Datele de test. Poate fi un array NumPy, o listă sau un șir de caractere care referențiază un nume de fișier din directorul de date partajate. | Da |
| `fit_params` | `dictionary` | Parametri suplimentari care vor fi pasați metodei `fit` a clasificatorului. | Nu |
| `options["save"]` | `boolean` | Dacă clasificatorul antrenat să fie salvat în directorul de date partajate. Implicit este `True`. | Nu |
| `options["out"]` | `str` | Numele fișierului JSON de ieșire în care să fie salvate predicțiile în directorul de date partajate. Dacă nu este furnizat, predicțiile sunt returnate în rezultatul job-ului. | Nu |

#### Utilizare

In [9]:
# delete the numpy files from your local disk
os.remove("X_train.npy")
os.remove("y_train.npy")
os.remove("X_test.npy")
os.remove("y_test.npy")

# delete the tar files from your local disk
os.remove("X_train.npy.tar")
os.remove("y_train.npy.tar")
os.remove("X_test.npy.tar")
os.remove("y_test.npy.tar")

# delete the tar files from the shared data
catalog.file_delete("X_train.npy.tar", singularity)
catalog.file_delete("y_train.npy.tar", singularity)
catalog.file_delete("X_test.npy.tar", singularity)
catalog.file_delete("y_test.npy.tar", singularity)

'Requested file was deleted.'

#### Validări
- `name`:
    - Dacă `options["save"]` este setat la `True`:
        - Numele trebuie să fie unic, un șir de caractere de până la 64 de caractere.
        - Poate conține doar caractere alfanumerice și underscore-uri.
        - Trebuie să înceapă cu o literă și nu poate să se termine cu un underscore.
        - Niciun clasificator cu același nume nu trebuie să existe deja în directorul de date partajate.

- `options["out"]`:
    - Numele fișierului trebuie să fie unic, un șir de caractere de până la 64 de caractere.
    - Poate conține doar caractere alfanumerice și underscore-uri.
    - Trebuie să înceapă cu o literă și nu poate să se termine cu un underscore.
    - Trebuie să aibă extensia `.json`.
---
## Primii pași
Autentifică-te folosind [cheia ta de API IBM Quantum Platform](http://quantum.cloud.ibm.com/) și selectează Qiskit Function după cum urmează:

In [10]:
# Import QiskitFunctionsCatalog to load the
# "Singularity Machine Learning - Classification" function by Multiverse Computing
from qiskit_ibm_catalog import QiskitFunctionsCatalog

# Import the make_moons and the train_test_split functions from scikit-learn
# to create a synthetic dataset and split it into training and test datasets
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

# authentication
# If you have not previously saved your credentials, follow instructions at
# /docs/guides/functions
# to authenticate with your API key.
catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# load "Singularity Machine Learning - Classification" function by Multiverse Computing
singularity = catalog.load("multiverse/singularity")

# generate the synthetic dataset
X, y = make_moons(n_samples=1000)

# split the data into training and test datasets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

job = singularity.run(
    action="create_fit_predict",
    num_learners=10,
    regularization=0.01,
    optimizer_options={"simulator": True},
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    options={"save": False},
)

# get job status and result
status = job.status()
result = job.result()

print("Job status: ", status)
print("Action result status: ", result["status"])
print("Action result message: ", result["message"])
print("Predictions (first five results): ", result["data"]["predictions"][:5])
print(
    "Probabilities (first five results): ",
    result["data"]["probabilities"][:5],
)
print("Usage metadata: ", result["metadata"]["resource_usage"])

Job status:  QUEUED
Action result status:  ok
Action result message:  Classifier created, fitted, and predicted.
Predictions (first five results):  [0, 0, 1, 0, 0]
Probabilities (first five results):  [[0.87119766531518, 0.1288023346848197], [0.87119766531518, 0.1288023346848197], [0.24470328446479797, 0.7552967155352032], [0.820524432250189, 0.17947556774981072], [0.6847610293419495, 0.31523897065805173]]
Usage metadata:  {'RUNNING: MAPPING': {'CPU_TIME': 10.967791318893433}, 'RUNNING: WAITING_QPU': {'CPU_TIME': 59.91712307929993}, 'RUNNING: POST_PROCESSING': {'CPU_TIME': 59.097386837005615}, 'RUNNING: EXECUTING_QPU': {'QPU_TIME': 56.93338203430176}}


## Exemplu
În acest exemplu, vei folosi funcția „Singularity Machine Learning - Classification" pentru a clasifica un set de date format din două semicercuri interconectate în formă de lună. Setul de date este sintetic, bidimensional și etichetat cu etichete binare. Este creat pentru a fi dificil pentru algoritmi precum clustering bazat pe centroizi și clasificare liniară.
![Setul de date Moons](../docs/images/guides/multiverse-computing-singularity/moon_shaped.avif)
Prin acest proces, vei învăța cum să creezi clasificatorul, să îl antrenezi pe datele de antrenament, să îl folosești pentru a face predicții pe datele de test și să îl ștergi când ai terminat.
Înainte de a începe, trebuie să instalezi [scikit-learn](https://scikit-learn.org/). Instalează-l folosind următoarea comandă:
2) Upload the generated synthetic dataset to the [shared data directory](https://qiskit.github.io/qiskit-serverless/function_features/experimental/manage_data_directory.html).
3) Create the quantum-enhanced classifier using the [`create`](https://docs.quantum.ibm.com/api/functions/multiverse-computing-singularity#create) action.
4) Enlist your classifiers using the [`list`](https://docs.quantum.ibm.com/api/functions/multiverse-computing-singularity#list) action.